# 01 - Data Exploration and Cleaning

**Goal:** understand the data, find errors, fix the clear ones, and list what the preprocessing pipeline must do.

Rules we follow:
- **Fixed here:** only clear errors that follow a fixed rule (empty rows, impossible values).
- **Left to the pipeline:** anything learned from the data (imputation, outliers, scaling, encoding). It must be fitted on train only, to avoid leakage.
- `spend_next_90d` is never used as a feature.
- A feature may only use the current row, the training data and `clientes.csv`.

Each section ends with a **Key insight**: what we found, and what to do in preprocessing.

Feature engineering and feature selection are in `02_feature_engineering_selection.ipynb`. This notebook saves the cleaned data for it at the end (Section 14).

## 1. Loading the data and understanding the columns

In [256]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = "../data"

In [257]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
clientes = pd.read_csv(f"{DATA_DIR}/clientes.csv")

print("train:   ", train.shape)
print("test:    ", test.shape)
print("clientes:", clientes.shape)

train:    (6534, 40)
test:     (986, 38)
clientes: (1143, 13)


In [258]:
# Which columns are in train but not in test (should be exactly the two targets)
print("Only in train:", sorted(set(train.columns) - set(test.columns)))
print("Only in test: ", sorted(set(test.columns) - set(train.columns)))

Only in train: ['repeat_purchase_90d', 'spend_next_90d']
Only in test:  []


In [259]:
train.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,repeat_purchase_90d,spend_next_90d,days_since_previous_purchase,customer_tenure_days,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,prior_std_purchase_value,prior_min_purchase_value,prior_max_purchase_value,prior_mean_gap_days,prior_std_gap_days,purchase_count_30d,spend_30d,purchase_count_90d,spend_90d,purchase_count_180d,spend_180d,purchase_count_365d,spend_365d,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,1,359.04,79.0,126.0,2.0,858.30,429.150000,112.656252,349.49,508.81,47.000000,NaN,0.0,0.00,1.0,349.49,2.0,858.30,2.0,858.30,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660448
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,1,592.71,244.0,616.0,5.0,6722.83,1344.566000,1683.795178,84.38,4222.15,93.000000,117.657696,0.0,0.00,0.0,0.00,0.0,0.00,4.0,2500.68,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,10000001,0.924510
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,1,423.72,9.0,610.0,90.0,17590.08,195.445333,190.389932,4.54,1002.15,6.752809,6.384073,2.0,111.27,9.0,1928.55,28.0,5290.25,49.0,7899.80,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,10000002,-0.258730
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,0,NaN,97.0,97.0,1.0,270.33,270.330000,NaN,270.33,270.33,NaN,NaN,0.0,0.00,0.0,0.00,1.0,270.33,1.0,270.33,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,10000003,1.448402
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,1,455.73,85.0,1575.0,25.0,12803.02,512.120800,406.696963,22.58,1736.74,62.083333,66.246520,0.0,0.00,2.0,184.13,4.0,2062.50,4.0,2062.50,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,10000004,-1.058890


### The columns

The brief groups the columns as below (Portuguese names translated). We save these groups in `column_groups` to use them later.

| Group | Columns | Meaning |
|---|---|---|
| Identifiers | `ID`, `customer_id` | Row ID; customer key to join `clientes.csv` |
| Targets | `repeat_purchase_90d`, `spend_next_90d` | Buys again within 90 days (0/1); how much they spend in those 90 days (EUR). Train only |
| Date | `purchase_date` | Anonymised date; order and intervals preserved |
| Current purchase: money | `Total a Pagar` (total to pay), `Total Bruto` (gross), `Total Liquido` (net), `Total Imp_` (tax), `Total Portes` (shipping), `Total Desc_ Global` (global discount), `Total Desc_ Linha` (line discounts), `Custo` (cost) | Values of this invoice |
| Current purchase: size | `N_ Detalhes` (number of invoice lines), `invoice_count_current_purchase` | |
| Current purchase: categories | `payment_type`, `salesperson`, `commercial_zone`, `transport_method`, `warehouse`, `document_series` | |
| History: timing | `days_since_previous_purchase`, `customer_tenure_days`, `prior_purchase_count`, `prior_mean_gap_days`, `prior_std_gap_days` | Recency, time as a customer, number of past purchases, gaps between past purchases |
| History: value | `prior_total_spend`, `prior_mean/std/min/max_purchase_value` | Summary of past purchase values |
| Recent windows: count | `purchase_count_30d/90d/180d/365d` | Purchases in the previous N days |
| Recent windows: spend | `spend_30d/90d/180d/365d` | Spend in the previous N days |
| Noise | `random_noise` | To check if it is useful |

In [260]:
column_groups = {
    "identifiers": ["ID", "customer_id"],
    "targets": ["repeat_purchase_90d", "spend_next_90d"],
    "date": ["purchase_date"],
    "current_monetary": ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_",
                         "Total Portes", "Total Desc_ Global", "Total Desc_ Linha", "Custo"],
    "current_details": ["N_ Detalhes", "invoice_count_current_purchase"],
    "categoricals": ["payment_type", "salesperson", "commercial_zone",
                     "transport_method", "warehouse", "document_series"],
    "hist_timing": ["days_since_previous_purchase", "customer_tenure_days", "prior_purchase_count",
                    "prior_mean_gap_days", "prior_std_gap_days"],
    "hist_value": ["prior_total_spend", "prior_mean_purchase_value", "prior_std_purchase_value",
                   "prior_min_purchase_value", "prior_max_purchase_value"],
    "rolling_count": ["purchase_count_30d", "purchase_count_90d", "purchase_count_180d", "purchase_count_365d"],
    "rolling_spend": ["spend_30d", "spend_90d", "spend_180d", "spend_365d"],
    "noise": ["random_noise"],
}

# Sanity check: every train column belongs to exactly one group
grouped = [c for cols in column_groups.values() for c in cols]
print("Columns in train but not in any group:", set(train.columns) - set(grouped))
print("Columns in a group but not in train:  ", set(grouped) - set(train.columns))
print("Columns listed twice:                 ", len(grouped) - len(set(grouped)))

Columns in train but not in any group: set()
Columns in a group but not in train:   set()
Columns listed twice:                  0


### Data types and missing values

One row per column: type in train and test, missing values, and number of distinct values. A column that is `int` in test but `float` in train usually has missing values in train (pandas turns integers with NaN into floats).

In [261]:
overview = pd.DataFrame({
    "dtype_train": train.dtypes,
    "dtype_test": test.dtypes.reindex(train.columns),
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum().reindex(train.columns),
    "missing_%_train": (train.isna().mean() * 100).round(1),
    "missing_%_test": (test.isna().mean().reindex(train.columns) * 100).round(1),
    "n_unique_train": train.nunique(),
})
overview

,dtype_train,dtype_test,missing_train,missing_test,missing_%_train,missing_%_test,n_unique_train
Total a Pagar,float64,float64,65,0.0,1.0,0.0,6122
Total Bruto,float64,float64,65,0.0,1.0,0.0,6038
Total Liquido,float64,float64,65,0.0,1.0,0.0,6093
Total Imp_,float64,float64,65,0.0,1.0,0.0,5406
Total Portes,float64,float64,65,0.0,1.0,0.0,215
Total Desc_ Global,float64,float64,65,0.0,1.0,0.0,109
Total Desc_ Linha,float64,float64,65,0.0,1.0,0.0,3519
Custo,float64,float64,65,0.0,1.0,0.0,5736
N_ Detalhes,float64,int64,65,0.0,1.0,0.0,171
invoice_count_current_purchase,float64,int64,65,0.0,1.0,0.0,5


In [262]:
# Columns whose dtype differs between train and test
overview[overview["dtype_train"] != overview["dtype_test"]].dropna(subset=["dtype_test"])

,dtype_train,dtype_test,missing_train,missing_test,missing_%_train,missing_%_test,n_unique_train
N_ Detalhes,float64,int64,65,0.0,1.0,0.0,171
invoice_count_current_purchase,float64,int64,65,0.0,1.0,0.0,5
customer_tenure_days,float64,int64,65,0.0,1.0,0.0,2121
prior_purchase_count,float64,int64,65,0.0,1.0,0.0,277
purchase_count_30d,float64,int64,65,0.0,1.0,0.0,12
purchase_count_90d,float64,int64,65,0.0,1.0,0.0,23
purchase_count_180d,float64,int64,65,0.0,1.0,0.0,38
purchase_count_365d,float64,int64,65,0.0,1.0,0.0,61


### clientes.csv

Extra table with one row per customer: location (anonymised) and the year they became a customer.

In [263]:
clientes.head()

,customer_id,country_or_region,commercial_region,island,municipality,city,postal_area,postal_code,street,address,customer_type,company_name,customer_since_year
0,842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,City CY417: Missing Cell Port,NaN,Postal code PC828: #REF! Delivery Route,NaN,Address AD1120: Cell Without a Home,NaN,Excel Is Not A Database Ltd 150,2018
1,770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,NaN,Postal code PC56: #REF! Delivery Route,NaN,Address AD62: Cell Without a Home,NaN,Excel Is Not A Database Ltd 1064,2021
2,827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC827: #REF! Delivery Route,Street ST170: PowerPoint New Walk,Address AD449: Cell Without a Home,NaN,Excel Is Not A Database Ltd 113,2016
3,494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD454: Cell Without a Home,NaN,Excel Is Not A Database Ltd 430,2015
4,429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,NaN,Postal area PA271: Hidden Sheet Sector,Postal code PC826: #REF! Delivery Route,NaN,Address AD453: Cell Without a Home,NaN,Excel Is Not A Database Ltd 673,2015


In [264]:
pd.DataFrame({
    "dtype": clientes.dtypes,
    "missing": clientes.isna().sum(),
    "missing_%": (clientes.isna().mean() * 100).round(1),
    "n_unique": clientes.nunique(),
})

,dtype,missing,missing_%,n_unique
customer_id,int64,0,0.0,1143
country_or_region,str,0,0.0,11
commercial_region,str,114,10.0,37
island,str,578,50.6,10
municipality,str,159,13.9,183
city,str,546,47.8,417
postal_area,str,71,6.2,271
postal_code,str,170,14.9,828
street,str,598,52.3,511
address,str,6,0.5,1120


### Key insight - Section 1

**Found**
- train and test have the same 38 predictors; train also has the two targets.
- In train, 65 values are missing in almost every column, always on the same rows (checked in Section 2). test has none.
- Those rows are why some integer columns show as `float` in train.
- `clientes.csv` has one row per customer. Most columns are anonymised addresses with many missing values, and `customer_type` is completely empty.

**For preprocessing**
- Nothing to fix in the column types: they match once the empty rows are removed (Section 2).
- From `clientes.csv`, drop `customer_type` and keep only the few useful columns (Section 11).

In [265]:
# Plot style used in every chart below
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(size=13), title=dict(font=dict(size=17)), height=450,
    margin=dict(t=80, l=60, r=30, b=50), hoverlabel=dict(bgcolor="white"),
    uniformtext=dict(minsize=11, mode="show")))
pio.templates.default = "plotly_white+eda"

BLUE, ORANGE, GREY, RED = "#2f6690", "#d9822b", "#9aa5b1", "#c0392b"
TARGET, SPEND = "repeat_purchase_90d", "spend_next_90d"

# Working copy: every fix below is applied to `df`; `train` stays untouched for comparison
df = train.copy()
df["purchase_date"] = pd.to_datetime(df["purchase_date"])
test["purchase_date"] = pd.to_datetime(test["purchase_date"])


def rate_by_bins(data, column, bins, labels=None, fig=None, row=None, col=None, title=None, xlabel=None):
    """Bar chart of the repeat-purchase rate per bin of `column`, with the row count on each bar.
    Draws into subplot (row, col) of `fig` when given, otherwise shows its own figure."""
    groups = pd.cut(data[column], bins=bins, labels=labels)
    summary = data.groupby(groups, observed=True)[TARGET].agg(["mean", "size"])
    own_figure = fig is None
    fig = go.Figure() if own_figure else fig
    pos = {} if own_figure else dict(row=row, col=col)
    overall = data[TARGET].mean()
    fig.add_bar(x=summary.index.astype(str), y=summary["mean"], marker_color=BLUE, showlegend=False,
                text=[f"n={n}" for n in summary["size"]], textposition="outside",
                hovertemplate="%{x}<br>repeat rate: %{y:.2f}<br>%{text}<extra></extra>", **pos)
    fig.add_hline(y=overall, line_dash="dash", line_color=ORANGE, **pos)
    fig.update_yaxes(range=[0, 1.1], title_text="repeat rate", **pos)
    fig.update_xaxes(title_text=xlabel or column, **pos)
    if own_figure:
        fig.update_layout(title=f"{title or f'Repeat rate by {column}'}<br><sup>dashed line = overall rate ({overall:.2f})</sup>")
        fig.show()
    return summary


def add_density(fig, values, color, name, row=1, col=1, showlegend=True):
    """Smoothed distribution (KDE) of `values` as a filled line in subplot (row, col)."""
    from scipy.stats import gaussian_kde
    values = values[np.isfinite(values)]
    grid = np.linspace(values.min(), values.max(), 200)
    r, g, b = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    fig.add_scatter(x=grid, y=gaussian_kde(values)(grid), mode="lines", fill="tozeroy", name=name,
                    line=dict(color=color, width=2), fillcolor=f"rgba({r},{g},{b},0.3)",
                    legendgroup=name, showlegend=showlegend, row=row, col=col)

## 2. Empty rows and duplicates

In [266]:
feature_cols = [c for c in train.columns if c not in column_groups["identifiers"] + column_groups["targets"]]
empty_rows = df[feature_cols].isna().all(axis=1)
print("Rows with every predictor missing:", empty_rows.sum())
print("Columns that are still filled in those rows:", df.loc[empty_rows].columns[df.loc[empty_rows].notna().any()].tolist())
print("Repeat rate in those rows:", round(df.loc[empty_rows, TARGET].mean(), 3))

Rows with every predictor missing: 65
Columns that are still filled in those rows: ['repeat_purchase_90d', 'customer_id', 'ID']
Repeat rate in those rows: 0.508


In [267]:
missing_before = train.isna().sum()
missing_after = train.loc[~empty_rows].isna().sum()
compare = pd.DataFrame({"with empty rows": missing_before, "without empty rows": missing_after})
compare = compare[compare["with empty rows"] > 0].sort_values("with empty rows")

fig = go.Figure()
fig.add_bar(y=compare.index, x=compare["with empty rows"], orientation="h", name="with empty rows", marker_color=GREY)
fig.add_bar(y=compare.index, x=compare["without empty rows"], orientation="h", name="without empty rows", marker_color=BLUE)
fig.update_layout(title="Missing values per column, before and after removing the 65 empty rows",
                  xaxis_title="number of missing values", barmode="group", height=750)
fig.show()

In [268]:
# Exact duplicates, ignoring the ID (which is unique by construction)
dup_all = df.drop(columns="ID").duplicated()
print("Duplicates ignoring ID (all rows):     ", dup_all.sum())
print("  ...of which are empty rows:          ", (dup_all & empty_rows).sum())

df = df.loc[~empty_rows].copy()
print("Duplicates ignoring ID (after removal):", df.drop(columns="ID").duplicated().sum())
print("Same customer, same date:              ", df.duplicated(subset=["customer_id", "purchase_date"]).sum())
print("Rows left:", len(df))

Duplicates ignoring ID (all rows):      10
  ...of which are empty rows:           10
Duplicates ignoring ID (after removal): 0
Same customer, same date:               0
Rows left: 6469


### Key insight - Section 2

**Found**
- 65 rows are empty: only `ID`, `customer_id` and the target are filled, so there is nothing to learn from them.
- The only duplicates (10) are among these empty rows. Once they are removed there are **no duplicates**, not even the same customer on the same day.
- So each row is one distinct purchase.

**For preprocessing**
- Drop the 65 empty rows from train (done here). test has none, so no prediction is lost.
- No deduplication step is needed.

## 3. Impossible values and sentinel codes

Some columns must follow rules by definition, so a value that breaks a rule is wrong:
- mean purchase value = total spend / number of purchases
- min <= mean <= max <= total spend
- a longer window contains a shorter one: `spend_30d` <= `spend_90d` <= `spend_180d` <= `spend_365d` <= total spend (same for counts)
- a number of days cannot be negative

The table shows the 10 columns whose maximum is furthest above the 99th percentile (`max / p99`), and how many negative values each has.

In [269]:
num_cols = df.select_dtypes("number").columns.drop(column_groups["identifiers"] + column_groups["targets"])
summary = df[num_cols].describe(percentiles=[0.01, 0.5, 0.99]).T
summary["max / p99"] = summary["max"] / summary["99%"]
summary["n_negative"] = (df[num_cols] < 0).sum()
summary.sort_values("max / p99", ascending=False).round(2).head(10)

,count,mean,std,min,1%,50%,99%,max,max / p99,n_negative
prior_mean_purchase_value,5672.0,2821825.32,53041566.66,-2478.66,42.35,724.90,5230.09,1.000000e+09,191201.24,16
spend_365d,6274.0,2396534.88,48841133.46,-159115.18,0.00,2426.60,40876.80,1.000000e+09,24463.76,15
Total Desc_ Global,6469.0,0.35,5.12,0.00,0.00,0.00,7.85,2.184900e+02,27.85,0
prior_max_purchase_value,5866.0,2946.38,5901.55,3.31,75.46,1783.24,14433.41,2.675262e+05,18.54,0
Total Portes,6469.0,1.15,5.84,0.00,0.00,0.00,11.80,1.810600e+02,15.34,0
prior_mean_gap_days,5226.0,95.88,122.59,-4.00,5.79,68.44,514.10,3.199000e+03,6.22,16
prior_std_gap_days,5056.0,74.14,101.82,0.00,4.87,50.64,486.34,2.212540e+03,4.55,0
spend_30d,6469.0,457.14,1191.23,0.00,0.00,0.00,5396.49,2.229721e+04,4.13,0
days_since_previous_purchase,5673.0,101.53,165.70,-4.00,1.00,57.00,787.12,3.199000e+03,4.06,16
Total Imp_,6469.0,152.22,225.26,0.00,0.00,79.74,1026.79,4.169400e+03,4.06,0


### Distribution of each numeric variable

Pick a variable in the dropdown (top right).
- **Top:** histogram of the middle 98% of values, to see the shape.
- **Bottom:** box plot of **all** values; points far from the box are extreme values (e.g. the `1e9` placeholders).

Hover for exact numbers, drag to zoom, double-click to reset.

In [270]:
fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3], vertical_spacing=0.15,
                    subplot_titles=("Histogram of the middle 98% (1%-99%)", "Box plot of all values"))
buttons = []
for i, variable in enumerate(num_cols):
    values = df[variable].dropna()
    low, high = values.quantile([0.01, 0.99])
    bulk = values[values.between(low, high)]
    fig.add_histogram(x=bulk, nbinsx=50, marker_color=BLUE, name=variable, visible=(i == 0), row=1, col=1)
    fig.add_box(x=values, marker_color=BLUE, boxpoints="outliers", name="", visible=(i == 0), row=2, col=1)

    title = (f"{variable}<br><sup>median {values.median():,.2f} | max {values.max():,.2f} | "
             f"negatives {(values < 0).sum()} | missing {df[variable].isna().sum()} | "
             f"outside 1%-99%: {len(values) - len(bulk)}</sup>")
    visible = [False] * (2 * len(num_cols))
    visible[2 * i] = visible[2 * i + 1] = True
    buttons.append(dict(label=variable, method="update", args=[{"visible": visible}, {"title.text": title}]))

fig.update_layout(title=buttons[0]["args"][1]["title.text"], height=600, showlegend=False,
                  updatemenus=[dict(buttons=buttons, x=1, xanchor="right", y=1.2, yanchor="top")])
fig.update_yaxes(title_text="number of rows", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.show()

Two columns stand out: `prior_mean_purchase_value` and `spend_365d` reach exactly **1,000,000,000** and have negative values. `1e9` is a placeholder (a *sentinel*), not a real amount.

### 3.1 `prior_mean_purchase_value`

In [271]:
has_history = df["prior_purchase_count"] > 0
recomputed_mean = df["prior_total_spend"] / df["prior_purchase_count"]
recomputed_mean = recomputed_mean.where(has_history)
stored_mean = df["prior_mean_purchase_value"]

sentinel = stored_mean == 1e9
negative = stored_mean < 0
matches = (stored_mean - recomputed_mean).abs() <= 0.01
inflated = stored_mean.notna() & ~sentinel & ~negative & ~matches & has_history

print("Matches total / count:", matches.sum())
print("Sentinel 1e9:         ", sentinel.sum())
print("Negative:             ", negative.sum(), "-> equal to minus the true mean:",
      np.allclose(-stored_mean[negative], recomputed_mean[negative]))
print("Inflated:             ", inflated.sum(), "-> stored / true ratio:",
      (stored_mean[inflated] / recomputed_mean[inflated]).describe()[["min", "50%", "max"]].round(1).to_dict())
print("Missing with history: ", (stored_mean.isna() & has_history).sum())

Matches total / count: 5594
Sentinel 1e9:          16
Negative:              16 -> equal to minus the true mean: True
Inflated:              46 -> stored / true ratio: {'min': 8.5, '50%': 16.1, 'max': 24.8}
Missing with history:  194


In [272]:
plot_df = pd.DataFrame({"true": recomputed_mean, "stored": stored_mean}).dropna()
plot_df = plot_df[(plot_df["stored"] > 0) & (plot_df["stored"] < 1e9)]
is_inflated = inflated.loc[plot_df.index]

lims = [1, 3e5]
fig = go.Figure()
fig.add_scatter(x=plot_df.loc[~is_inflated, "true"], y=plot_df.loc[~is_inflated, "stored"], mode="markers",
                marker=dict(size=5, color=BLUE, opacity=0.4), name="consistent")
fig.add_scatter(x=plot_df.loc[is_inflated, "true"], y=plot_df.loc[is_inflated, "stored"], mode="markers",
                marker=dict(size=8, color=RED), name=f"inflated ({is_inflated.sum()})")
fig.add_scatter(x=lims, y=lims, mode="lines", line=dict(color=GREY, dash="dash", width=1), name="stored = true")
fig.update_xaxes(type="log", range=np.log10(lims), title_text="prior_total_spend / prior_purchase_count")
fig.update_yaxes(type="log", range=np.log10(lims), title_text="stored prior_mean_purchase_value")
fig.update_layout(title="Stored mean vs mean recomputed from total and count", width=700, height=600)
fig.show()

Almost every point lies on the diagonal (stored = true). The errors are:
- 16 placeholders (`1e9`)
- 16 sign flips (exactly minus the true value)
- 46 values 8 to 25 times too large

**Fix:** total spend and count are always available, so we recompute the mean as total / count. This fixes all 78 errors and also fills the 194 missing values.

In [273]:
df["prior_mean_purchase_value"] = recomputed_mean
check = df["prior_mean_purchase_value"]
print("Mean outside [min, max] after the fix:",
      ((check < df["prior_min_purchase_value"] - 0.01) | (check > df["prior_max_purchase_value"] + 0.01)).sum())

Mean outside [min, max] after the fix: 0


### 3.2 `prior_max_purchase_value`

In [274]:
max_too_big = df["prior_max_purchase_value"] > df["prior_total_spend"] + 0.01
print("prior_max > prior_total_spend:", max_too_big.sum())
print("ratio max / total in those rows:",
      (df.loc[max_too_big, "prior_max_purchase_value"] / df.loc[max_too_big, "prior_total_spend"]).describe()[["min", "50%", "max"]].round(1).to_dict())
print("prior_min > prior_mean:", (df["prior_min_purchase_value"] > df["prior_mean_purchase_value"] + 0.01).sum())

prior_max > prior_total_spend: 45
ratio max / total in those rows: {'min': 1.1, '50%': 4.3, 'max': 22.3}
prior_min > prior_mean: 0


One purchase cannot be worth more than all purchases together, so these 45 values are wrong (1.1 to 22 times the total).

**Fix:** rebuild the max from the client's earlier purchases in train (the biggest `Total a Pagar` before this one).
- If 1 or 2 earlier purchases are missing from train, their total is `prior_total_spend` minus what we see. If that amount is smaller than the max we see, it can't change the max.
- Anything that still can't be rebuilt becomes NaN and is imputed in the pipeline.

In [275]:
def rebuild_max(data):
    """Max of the client's earlier Total a Pagar in this data. NaN when it can't be known for sure."""
    ordered = data.sort_values(["customer_id", "purchase_date"])
    g = ordered.groupby("customer_id")["Total a Pagar"]
    seen_max = g.transform(lambda s: s.shift().cummax())      # biggest earlier purchase we can see
    seen_sum = g.transform(lambda s: s.shift().cumsum())      # total of earlier purchases we can see
    seen_n = ordered.groupby("customer_id").cumcount()        # how many earlier purchases we can see
    missing_n = ordered["prior_purchase_count"] - seen_n      # earlier purchases not in the data
    missing_sum = ordered["prior_total_spend"] - seen_sum.fillna(0)
    rebuilt = seen_max.where(missing_n == 0)                                        # full history
    rebuilt = rebuilt.fillna(seen_max.where((missing_n > 0) & (missing_sum <= seen_max + 0.01)))  # missing ones too small to matter
    return rebuilt.reindex(data.index)

rebuilt_max = rebuild_max(df)
stored_max = df["prior_max_purchase_value"]

# Proof the method works: on rows where the stored max is correct, does the rebuilt max match it?
correct = ~max_too_big & stored_max.between(0, 1e8) & rebuilt_max.notna()
print("Method check on correct rows:", ((rebuilt_max - stored_max).abs() <= 0.01)[correct].mean().round(3), "match")

df.loc[max_too_big, "prior_max_purchase_value"] = rebuilt_max[max_too_big]
print("prior_max rebuilt:", (max_too_big & rebuilt_max.notna()).sum(), "of", max_too_big.sum(),
      "| still NaN (imputed later):", (max_too_big & rebuilt_max.isna()).sum())

Method check on correct rows: 0.999 match
prior_max rebuilt: 45 of 45 | still NaN (imputed later): 0


### 3.3 Rolling spend and count windows

In [276]:
windows = ["30d", "90d", "180d", "365d"]
rows = []
for short, long in zip(windows[:-1], windows[1:]):
    rows.append({"rule": f"count_{short} <= count_{long}",
                 "violations": (df[f"purchase_count_{short}"] > df[f"purchase_count_{long}"]).sum()})
    rows.append({"rule": f"spend_{short} <= spend_{long}",
                 "violations": (df[f"spend_{short}"] > df[f"spend_{long}"] + 0.01).sum()})
rows.append({"rule": "count_365d <= prior_purchase_count",
             "violations": (df["purchase_count_365d"] > df["prior_purchase_count"]).sum()})
rows.append({"rule": "spend_365d <= prior_total_spend",
             "violations": (df["spend_365d"] > df["prior_total_spend"] + 0.01).sum()})
pd.DataFrame(rows)

,rule,violations
0,count_30d <= count_90d,0
1,spend_30d <= spend_90d,0
2,count_90d <= count_180d,0
3,spend_90d <= spend_180d,0
4,count_180d <= count_365d,0
5,spend_180d <= spend_365d,15
6,count_365d <= prior_purchase_count,0
7,spend_365d <= prior_total_spend,54


In [277]:
s365 = df["spend_365d"]
print("Sentinel 1e9:", (s365 == 1e9).sum())
print("Negative:    ", (s365 < 0).sum())
# The other two columns of the rule have no negatives, so the error is in spend_365d
print("  negatives in spend_180d:", (df["spend_180d"] < 0).sum(), "| in prior_total_spend:", (df["prior_total_spend"] < 0).sum())
# ...and the 15 negatives are exactly the 15 rows that break spend_180d <= spend_365d
breaks_180 = df["spend_180d"] > s365 + 0.01
print("  rows breaking spend_180d <= spend_365d:", breaks_180.sum(), "| of which spend_365d is negative:", (breaks_180 & (s365 < 0)).sum())
flipped = -s365[s365 < 0]
print("  |value| fits between spend_180d and prior_total_spend:",
      ((flipped >= df.loc[flipped.index, "spend_180d"] - 0.01) &
       (flipped <= df.loc[flipped.index, "prior_total_spend"] + 0.01)).sum(), "of", len(flipped))
print("Larger than prior_total_spend (not 1e9):", ((s365 > df["prior_total_spend"] + 0.01) & (s365 != 1e9)).sum())

Sentinel 1e9: 15
Negative:     15
  negatives in spend_180d: 0 | in prior_total_spend: 0
  rows breaking spend_180d <= spend_365d: 15 | of which spend_365d is negative: 15
  |value| fits between spend_180d and prior_total_spend: 14 of 15
Larger than prior_total_spend (not 1e9): 39


In [278]:
valid = df[(df["spend_365d"] >= 0) & (df["spend_365d"] < 1e9)]
too_big = valid["spend_365d"] > valid["prior_total_spend"] + 0.01

fig = go.Figure()
fig.add_scatter(x=valid.loc[~too_big, "prior_total_spend"] + 1, y=valid.loc[~too_big, "spend_365d"] + 1, mode="markers",
                marker=dict(size=5, color=BLUE, opacity=0.4), name="valid")
fig.add_scatter(x=valid.loc[too_big, "prior_total_spend"] + 1, y=valid.loc[too_big, "spend_365d"] + 1, mode="markers",
                marker=dict(size=8, color=RED), name=f"spend_365d > total ({too_big.sum()})")
fig.add_scatter(x=[1, 1e6], y=[1, 1e6], mode="lines", line=dict(color=GREY, dash="dash", width=1), name="equal")
fig.update_xaxes(type="log", title_text="prior_total_spend + 1")
fig.update_yaxes(type="log", title_text="spend_365d + 1")
fig.update_layout(title="Spend in the last year cannot exceed all-time spend", width=700, height=600)
fig.show()

Only `spend_365d` breaks the rules. Its errors:
- 15 placeholders (`1e9`)
- 15 negative values. `spend_180d` and total spend have no negatives, and these 15 rows are exactly the 15 that break `spend_180d` <= `spend_365d`, so the error is in `spend_365d`. 14 of them become valid when the sign is removed (they fit between `spend_180d` and total spend)
- 39 values larger than all-time spend

**Fix:**
1. Remove the sign.
2. Keep the value only if `spend_180d` <= `spend_365d` <= total spend. The other 55 are wrong.
3. Rebuild the wrong ones from dates: the sum of the client's `Total a Pagar` in the 365 days before this purchase (in train). This only works when all the client's earlier purchases are in train. A purchase hidden in an empty row has no date, so we can't know if it falls inside the 365 days.
4. Wrong values that can't be rebuilt become NaN and are imputed in the pipeline.

In [279]:
def rebuild_spend_365d(data):
    """Sum of the client's Total a Pagar in the 365 days before each purchase. NaN if some earlier purchase is hidden."""
    rebuilt = pd.Series(np.nan, index=data.index)
    for _, client in data.groupby("customer_id"):
        client = client.sort_values("purchase_date")
        dates, paid = client["purchase_date"].values, client["Total a Pagar"].values
        for i, row in enumerate(client.index):
            if client.loc[row, "prior_purchase_count"] != i:              # some earlier purchase is hidden
                continue
            days_before = (dates[i] - dates[:i]) / np.timedelta64(1, "D")  # earlier purchases only
            rebuilt[row] = paid[:i][days_before <= 365].sum()
    return rebuilt

df["spend_365d"] = df["spend_365d"].abs()                                 # 1. remove the sign
valid_365 = (df["spend_365d"] >= df["spend_180d"] - 0.01) & (df["spend_365d"] <= df["prior_total_spend"] + 0.01)
wrong_365 = df["spend_365d"].notna() & ~valid_365                          # 2. the 55 wrong values

rebuilt_365 = rebuild_spend_365d(df)
# Proof the method works: on correct rows, does the rebuilt value match the stored one?
correct = valid_365 & rebuilt_365.notna() & (df["prior_purchase_count"] > 0)
print("Method check on correct rows:", ((rebuilt_365 - df["spend_365d"]).abs() <= 0.01)[correct].mean().round(3), "match")

df.loc[wrong_365, "spend_365d"] = rebuilt_365[wrong_365]                   # 3. real value, or 4. NaN if unknown
print("spend_365d wrong:", wrong_365.sum(),
      "| rebuilt:", (wrong_365 & rebuilt_365.notna()).sum(),
      "| set to NaN (imputed later):", (wrong_365 & rebuilt_365.isna()).sum())

Method check on correct rows: 1.0 match
spend_365d wrong: 55 | rebuilt: 45 | set to NaN (imputed later): 10


### 3.4 Negative day counts

In [280]:
gap_cols = ["days_since_previous_purchase", "prior_mean_gap_days"]
fig = make_subplots(rows=1, cols=2, subplot_titles=[f"{col} (values <= 15)" for col in gap_cols])
for i, col in enumerate(gap_cols, start=1):
    counts = df[col].clip(upper=15).round().value_counts().sort_index()
    fig.add_bar(x=counts.index, y=counts.values, marker_color=[RED if v < 0 else BLUE for v in counts.index],
                showlegend=False, hovertemplate="%{x} days: %{y} rows<extra></extra>", row=1, col=i)
    fig.update_xaxes(title_text="days (15 = 15 or more)", row=1, col=i)
fig.update_layout(height=420)
fig.show()

for col in gap_cols:
    print(col, "negative values:", df.loc[df[col] < 0, col].value_counts().to_dict())

days_since_previous_purchase negative values: {-4.0: 16}
prior_mean_gap_days negative values: {-4.0: 16}


In [281]:
# Is -4 a real value? Compare with the gap to the customer's previous purchase in train
ordered = df.sort_values(["customer_id", "purchase_date"])
gap_in_train = ordered.groupby("customer_id")["purchase_date"].diff().dt.days
compare_gap = pd.DataFrame({"stored": ordered["days_since_previous_purchase"], "gap_in_train": gap_in_train}).dropna()

print("stored == gap computed from dates:", (compare_gap["stored"] == compare_gap["gap_in_train"]).mean().round(3))
print("gap in train for the rows stored as -4:", compare_gap.loc[compare_gap["stored"] < 0, "gap_in_train"].tolist())

# When they disagree (ignoring the -4), which one is bigger?
mismatch = compare_gap[(compare_gap["stored"] >= 0) & (compare_gap["stored"] != compare_gap["gap_in_train"])]
print("rows where they disagree:", len(mismatch),
      "| gap from dates is bigger:", (mismatch["gap_in_train"] > mismatch["stored"]).sum(),
      "| gap from dates is smaller:", (mismatch["gap_in_train"] < mismatch["stored"]).sum())

stored == gap computed from dates: 0.988
gap in train for the rows stored as -4: [3.0, 231.0, 350.0, 13.0, 77.0, 25.0, 21.0, 14.0, 112.0, 47.0, 279.0, 32.0, 63.0, 35.0, 111.0, 105.0]
rows where they disagree: 51 | gap from dates is bigger: 51 | gap from dates is smaller: 0


The stored gap equals the gap computed from the purchase dates in **98.8%** of rows. For the 16 rows stored as **-4**, the real gaps go from 3 to 350 days, so -4 is a placeholder, not a measurement. The same -4 appears 16 times in `prior_mean_gap_days`.

**Fix:**
- `days_since_previous_purchase`: replace -4 with the gap computed from the dates. In the 51 rows where the two disagree, the date gap is always the larger one (the real previous purchase is missing from train), so a few rebuilt values may be too large. Still better than an imputed guess.
- `prior_mean_gap_days`: the average gap between past purchases = (date of the last past purchase - date of the first purchase) / (past purchases - 1). Rebuild it from the dates when all the client's earlier purchases are in train. If one is hidden in an empty row, -4 becomes NaN and is imputed.

The 98.8% match also shows that train has almost the full purchase history of each customer (used again in Sections 6 and 11).

In [282]:
# days_since_previous_purchase: replace -4 with the real gap, rebuilt from the customer's previous purchase date in train
fill = compare_gap.index[compare_gap["stored"] < 0]
df.loc[fill, "days_since_previous_purchase"] = compare_gap.loc[fill, "gap_in_train"]
print("days_since_previous_purchase rebuilt from dates:", len(fill),
      "| still negative:", (df["days_since_previous_purchase"] < 0).sum())

def rebuild_mean_gap(data):
    """(last past purchase - first purchase) / (past purchases - 1). NaN if some earlier purchase is hidden."""
    ordered = data.sort_values(["customer_id", "purchase_date"])
    g = ordered.groupby("customer_id")["purchase_date"]
    seen_n = ordered.groupby("customer_id").cumcount()                    # earlier purchases we can see
    span = (g.shift() - g.transform("min")).dt.days                        # last past purchase - first purchase
    rebuilt = (span / (seen_n - 1)).where((ordered["prior_purchase_count"] == seen_n) & (seen_n >= 2))
    return rebuilt.reindex(data.index)

# prior_mean_gap_days: replace -4 with the real mean gap, rebuilt from the purchase dates in train
rebuilt_gap = rebuild_mean_gap(df)
correct = (df["prior_mean_gap_days"] >= 0) & rebuilt_gap.notna()
print("Method check on correct rows:", ((rebuilt_gap - df["prior_mean_gap_days"]).abs() <= 0.01)[correct].mean().round(3), "match")
minus4 = df["prior_mean_gap_days"] < 0
df.loc[minus4, "prior_mean_gap_days"] = rebuilt_gap[minus4]               # NaN when it can't be rebuilt
print("prior_mean_gap_days -4:", minus4.sum(), "| rebuilt:", (minus4 & rebuilt_gap.notna()).sum(),
      "| set to NaN (imputed later):", (minus4 & rebuilt_gap.isna()).sum())

days_since_previous_purchase rebuilt from dates: 16 | still negative: 0
Method check on correct rows: 1.0 match
prior_mean_gap_days -4: 16 | rebuilt: 15 | set to NaN (imputed later): 1


### Key insight - Section 3

**Found**

| Column | Problem | Rows | Fix here |
|---|---|---|---|
| `prior_mean_purchase_value` | `1e9`, sign flip, 8-25x too large, missing | 16 + 16 + 46 + 194 | recompute as total / count (exact) |
| `prior_max_purchase_value` | larger than total spend | 45 | biggest earlier purchase in train (all 45) |
| `spend_365d` | `1e9`, sign flip, larger than total spend | 15 + 15 + 39 | remove sign; the 55 still invalid are rebuilt from dates (45), the rest NaN (10) |
| `days_since_previous_purchase` | -4 placeholder | 16 | real gap from the purchase dates |
| `prior_mean_gap_days` | -4 placeholder | 16 | real mean gap from the purchase dates (15), NaN (1) |

The same errors appear in test too (2 x `1e9` in each of the two columns, 2 x -4 in each gap column).

**For preprocessing**
- Apply the same rules to test: they only use the row itself. Exception: the rebuilds from dates (max, `spend_365d`, -4 gaps) are done in train only; in test these values become NaN.
- Impute the NaN created here (`prior_max_purchase_value`, `spend_365d`, `prior_mean_gap_days`) inside the pipeline, fitted on train only (e.g. median).
- Test a missing-value indicator for these columns, so the model knows the value was estimated.

## 4. Consistency of the monetary columns

In [283]:
# Each invoice should add up: compute "left side - right side" for each rule (0 means the rule holds)
checks = {
    # gross value minus both discounts should give the net value
    "Bruto - Desc_Linha - Desc_Global = Liquido": df["Total Bruto"] - df["Total Desc_ Linha"] - df["Total Desc_ Global"] - df["Total Liquido"],
    # net value plus tax should give the total to pay
    "Liquido + Imp = a Pagar": df["Total Liquido"] + df["Total Imp_"] - df["Total a Pagar"],
    # same, but also adding shipping: tests whether shipping is included in the total
    "Liquido + Imp + Portes = a Pagar": df["Total Liquido"] + df["Total Imp_"] + df["Total Portes"] - df["Total a Pagar"],
}
# Share of rows where each rule holds; 0.02 EUR of tolerance absorbs rounding to cents
pd.DataFrame({name: {"rows that hold (|diff| <= 0.02)": (diff.abs() <= 0.02).mean().round(4)} for name, diff in checks.items()}).T

,rows that hold (|diff| <= 0.02)
Bruto - Desc_Linha - Desc_Global = Liquido,1.0000
Liquido + Imp = a Pagar,1.0000
Liquido + Imp + Portes = a Pagar,0.8372


### Key insight - Section 4

**Found**
- The invoice arithmetic is exact in 100% of rows: gross - discounts = net, and net + tax = total to pay.
- Shipping is **not** part of the total, which is why the third rule fails in the 1,053 rows with shipping.

**For preprocessing**
- Nothing to fix: the monetary columns are consistent with each other.
- Ratios built from these columns (VAT rate, margin, cost = 0) are explored in notebook 02.

## 5. Missing values: structural or random?

In [284]:
first_purchase = df["prior_purchase_count"] == 0
one_prior = df["prior_purchase_count"] == 1
two_prior = df["prior_purchase_count"] == 2
# Each customer has at most one 1st and one 2nd purchase, so these also count customers
print("Customers in train:", df["customer_id"].nunique(),
      "| with their 1st purchase in train:", df.loc[first_purchase, "customer_id"].nunique(),
      "| with a 2nd purchase in train:", df.loc[one_prior, "customer_id"].nunique())

hist_cols = column_groups["hist_timing"] + column_groups["hist_value"] + ["spend_365d"]
# Which NaN are expected by definition?
needs_1_past = [c for c in hist_cols if c != "spend_365d"]                     # 1st purchase: no history (spend_365d should be 0, not NaN)
needs_2_past = ["prior_std_purchase_value", "prior_std_gap_days", "prior_mean_gap_days"]  # std and gaps need 2 past purchases
needs_3_past = ["prior_std_gap_days"]                                          # std of the gaps needs 2 gaps = 3 past purchases

miss = pd.DataFrame({"missing": df[hist_cols].isna().sum()})
miss["expected: 1st purchase"] = df.loc[first_purchase, hist_cols].isna().sum().where(miss.index.isin(needs_1_past), 0)
miss["expected: 2nd purchase"] = df.loc[one_prior, hist_cols].isna().sum().where(miss.index.isin(needs_2_past), 0)
miss["expected: 3rd purchase"] = df.loc[two_prior, hist_cols].isna().sum().where(miss.index.isin(needs_3_past), 0)
miss["unexplained"] = miss["missing"] - miss.iloc[:, 1:].sum(axis=1)
miss[miss["missing"] > 0]

Customers in train: 607 | with their 1st purchase in train: 603 | with a 2nd purchase in train: 445


,missing,expected: 1st purchase,expected: 2nd purchase,expected: 3rd purchase,unexplained
days_since_previous_purchase,796,603,0,0,193
prior_mean_gap_days,1244,603,445,0,196
prior_std_gap_days,1413,603,445,365,0
prior_mean_purchase_value,603,603,0,0,0
prior_std_purchase_value,1048,603,445,0,0
prior_min_purchase_value,603,603,0,0,0
prior_max_purchase_value,603,603,0,0,0
spend_365d,205,0,0,0,205


In [285]:
cat_cols = column_groups["categoricals"]
cat_missing = df[cat_cols].isna()
print("Missing per categorical:", cat_missing.sum().to_dict())
print("Rows with ALL of payment/salesperson/transport/warehouse missing:",
      cat_missing[["payment_type", "salesperson", "transport_method", "warehouse"]].all(axis=1).sum())
print("Same in test:", test[["payment_type", "salesperson", "transport_method", "warehouse"]].isna().all(axis=1).sum())

Missing per categorical: {'payment_type': 195, 'salesperson': 194, 'commercial_zone': 0, 'transport_method': 196, 'warehouse': 195, 'document_series': 0}
Rows with ALL of payment/salesperson/transport/warehouse missing: 0
Same in test: 0


### Key insight - Section 5

**Found** - most missing values are missing *by definition*:
1. **1st purchase (603 rows):** no history, so recency, mean/min/max and gaps don't exist.
2. **2nd purchase (445 rows):** one past purchase is not enough for a standard deviation or for a gap between past purchases (`prior_std_*`, `prior_mean_gap_days`).
3. **3rd purchase (365 rows):** two past purchases give only one gap, so its standard deviation (`prior_std_gap_days`) doesn't exist.
4. **Random:** about 195 values are missing for no reason in `days_since_previous_purchase`, `prior_mean_gap_days` and `spend_365d` (the same number as in the categoricals). The rest of the `unexplained` column are the values set to NaN in Section 3.

- Of the 607 customers in train, 603 have their 1st purchase in the data and 445 came back at least once (about 74%).
- `payment_type`, `salesperson`, `transport_method` and `warehouse` have about 195 missing values each (30 in test), almost never on the same row.

**For preprocessing**
- Missing *by definition* means "no history", not "unknown value". Add an `is_first_purchase` flag (and use `prior_purchase_count` to mark 2nd/3rd purchases), then fill these NaN with a constant. The flag tells the model what the constant means.
- Random NaN: impute with the median (numeric) or a "missing" category (categorical), fitted on train only.
- Tree models such as HistGradientBoosting can also handle NaN directly; compare both options in validation.

## 6. The two targets

In [286]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("repeat_purchase_90d", "spend_next_90d (EUR)",
                                                    "log(1 + spend_next_90d)"))
counts = df[TARGET].value_counts().sort_index()
fig.add_bar(x=["0 - no repeat", "1 - repeat"], y=counts.values, marker_color=[GREY, BLUE],
            text=[f"{v} ({v / len(df):.0%})" for v in counts.values], textposition="outside", row=1, col=1)

returned = df[df[TARGET] == 1]
fig.add_histogram(x=returned[SPEND].dropna(), nbinsx=60, marker_color=BLUE, row=1, col=2)
fig.add_histogram(x=np.log1p(returned[SPEND].dropna()), nbinsx=60, marker_color=BLUE, row=1, col=3)
fig.update_xaxes(title_text="EUR", row=1, col=2)
fig.update_xaxes(title_text="log EUR", row=1, col=3)
fig.update_yaxes(range=[0, counts.max() * 1.15], row=1, col=1)
fig.update_layout(showlegend=False, height=420)
fig.show()

print("Skewness raw:", round(returned[SPEND].skew(), 2), "| log:", round(np.log1p(returned[SPEND]).skew(), 2))
print("Spend present when target = 0:", df.loc[df[TARGET] == 0, SPEND].notna().sum())
print("Spend missing when target = 1:", returned[SPEND].isna().sum())

Skewness raw: 3.04 | log: -0.57
Spend present when target = 0: 0
Spend missing when target = 1: 196


**Classification target:** 56% repeat, 44% don't. That is balanced enough, so no resampling is needed. Always answering "yes" already gives 56% accuracy, so ROC-AUC or F1 are better to compare models (check which metric Kaggle uses).

**Regression target:** very skewed (skewness 3.0). After `log1p` it is almost symmetric (-0.6).

196 returning customers have no spend value. The next cell checks if it can be recovered.

In [287]:
# spend_next_90d should be the sum of the customer's purchases in the 90 days after the current one
ordered = df.sort_values(["customer_id", "purchase_date"])


def future_spend(group, days=90):
    dates = group["purchase_date"].values
    values = group["Total a Pagar"].values
    result = []
    for d in dates:
        in_window = (dates > d) & (dates <= d + np.timedelta64(days, "D"))
        result.append(values[in_window].sum())
    return pd.Series(result, index=group.index)


df["spend_rebuilt"] = ordered.groupby("customer_id", group_keys=False).apply(future_spend)

known = df[SPEND].notna()
exact = (df.loc[known, "spend_rebuilt"] - df.loc[known, SPEND]).abs() <= 0.01 * df.loc[known, SPEND]
print("Rebuilt spend matches the real target (within 1%):", exact.mean().round(3))

fig = go.Figure()
fig.add_scatter(x=df.loc[known, SPEND] + 1, y=df.loc[known, "spend_rebuilt"] + 1, mode="markers",
                marker=dict(size=4, color=BLUE, opacity=0.4), showlegend=False)
fig.add_scatter(x=[1, 3e4], y=[1, 3e4], mode="lines", line=dict(color=GREY, dash="dash"), showlegend=False)
fig.update_xaxes(type="log", title_text="spend_next_90d given (+1)")
fig.update_yaxes(type="log", title_text="sum of next 90 days of purchases in train (+1)")
fig.update_layout(title="The spend target can be rebuilt from train", width=650, height=600)
fig.show()

missing_spend = (df[TARGET] == 1) & df[SPEND].isna()
print("Missing spend values that can be recovered (> 0):", (df.loc[missing_spend, "spend_rebuilt"] > 0).sum(), "of", missing_spend.sum())

Rebuilt spend matches the real target (within 1%): 0.971


Missing spend values that can be recovered (> 0): 176 of 196


### Key insight - Section 6

**Found**
- In 97% of rows, `spend_next_90d` is the sum of `Total a Pagar` of the customer's purchases in the next 90 days.
- So 176 of the 196 missing spend values can be recovered from train (done in the next cell). 20 cannot.

**For preprocessing**
- Regression: train with the 176 recovered values and drop the 20 still missing. Simpler alternative: drop all 196. This is a group decision; mention it in the report.
- ⚠️ The recovery uses future purchases. It may only fill the target, never be used as a feature.
- Train the regression on `log1p(spend)` and convert predictions back with `expm1`.
- Classes are balanced: no resampling or class weights needed.

In [288]:
recoverable = missing_spend & (df["spend_rebuilt"] > 0)
df.loc[recoverable, SPEND] = df.loc[recoverable, "spend_rebuilt"]
df = df.drop(columns="spend_rebuilt")
print("Returning customers still without spend:", ((df[TARGET] == 1) & df[SPEND].isna()).sum())

Returning customers still without spend: 20


## 7. Time

In [289]:
quarterly_train = df.set_index("purchase_date").resample("QE").size()
quarterly_test = test.set_index("purchase_date").resample("QE").size()
rate = df.set_index("purchase_date").resample("QE")[TARGET].mean()

# Two panels with a shared time axis (instead of two y-axes on one chart)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4], vertical_spacing=0.1,
                    subplot_titles=("Purchases per quarter", "Repeat rate per quarter (train)"))
bar_width = 80 * 24 * 3600 * 1000  # 80 days, in milliseconds
fig.add_bar(x=quarterly_train.index, y=quarterly_train.values, width=bar_width, marker_color=BLUE, name="train", row=1, col=1)
fig.add_bar(x=quarterly_test.index, y=quarterly_test.values, width=bar_width, marker_color=ORANGE, name="test", row=1, col=1)
fig.add_scatter(x=rate.index, y=rate.values, mode="lines+markers", line=dict(color=BLUE, width=2),
                marker=dict(size=8), name="repeat rate", showlegend=False, row=2, col=1)
fig.update_yaxes(title_text="purchases", row=1, col=1)
fig.update_yaxes(title_text="repeat rate", range=[0, 1], row=2, col=1)
fig.update_layout(title="Purchases over time: test comes strictly after train", height=600)
fig.show()

print("train:", df["purchase_date"].min().date(), "->", df["purchase_date"].max().date())
print("test: ", test["purchase_date"].min().date(), "->", test["purchase_date"].max().date())

train: 2015-07-11 -> 2025-02-15
test:  2025-05-24 -> 2026-10-06


In [290]:
yearly = df.groupby(df["purchase_date"].dt.year)[TARGET].agg(["mean", "size"])
monthly = df.groupby(df["purchase_date"].dt.month)[TARGET].mean()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Repeat rate per year", "Repeat rate by month of the current purchase"))
fig.add_scatter(x=yearly.index, y=yearly["mean"], mode="lines+markers", line=dict(color=BLUE, width=2),
                marker=dict(size=8), customdata=yearly["size"],
                hovertemplate="%{x}: %{y:.2f} (n=%{customdata})<extra></extra>", row=1, col=1)
fig.add_bar(x=monthly.index, y=monthly.values, marker_color=BLUE, row=1, col=2)
fig.add_hline(y=df[TARGET].mean(), line_dash="dash", line_color=ORANGE, row=1, col=2)
fig.update_yaxes(title_text="repeat rate", range=[0.3, 0.8], row=1, col=1)
fig.update_yaxes(range=[0, 1], row=1, col=2)
fig.update_xaxes(dtick=1, row=1, col=1)
fig.update_xaxes(tickmode="linear", tick0=1, dtick=1, row=1, col=2)
fig.update_layout(showlegend=False, height=420)
fig.show()

### Key insight - Section 7

**Found**
- Train covers Jul 2015 - Feb 2025 and test covers May 2025 - Oct 2026. No overlap: we are predicting the future.
- The repeat rate per year moves between 0.48 and 0.63, with no trend.
- There is a month effect: March and November about 0.45, January, May and September about 0.70.

**For preprocessing and validation**
- Use **time-based validation**: train on older data and validate on the most recent 15-20% (or `TimeSeriesSplit`). Random K-fold would give scores that are too optimistic.
- Test `month` as a feature (one-hot, or sine/cosine). Dates are anonymised, so keep it only if it helps in validation.
- Don't use the date itself as a number: test dates are outside the train range.

## 8. What drives a repeat purchase?

In [291]:
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.18, horizontal_spacing=0.08,
                    subplot_titles=("Recency: days since the previous purchase", "Frequency: number of prior purchases",
                                    "Purchases in the previous 90 days", "Value of the current purchase"))
rate_by_bins(df, "days_since_previous_purchase", [0, 7, 14, 30, 60, 90, 180, 365, 5000], fig=fig, row=1, col=1,
             xlabel="days")
rate_by_bins(df, "prior_purchase_count", [-1, 0, 1, 2, 5, 10, 25, 50, 100, 300], fig=fig, row=1, col=2,
             labels=["0", "1", "2", "3-5", "6-10", "11-25", "26-50", "51-100", "100+"], xlabel="prior purchases")
rate_by_bins(df, "purchase_count_90d", [-1, 0, 1, 2, 3, 5, 10, 30], fig=fig, row=2, col=1,
             labels=["0", "1", "2", "3", "4-5", "6-10", "10+"], xlabel="purchases")
rate_by_bins(df, "Total a Pagar", [0, 50, 150, 400, 1000, 3000, 30000], fig=fig, row=2, col=2, xlabel="EUR")
fig.update_layout(title=f"Repeat rate by recency, frequency and value<br><sup>dashed line = overall rate ({df[TARGET].mean():.2f})</sup>",
                  height=900, margin_t=120)
fig.show()

- **Frequency** is the strongest signal: 1 prior purchase -> 37% repeat, 100+ -> 99%.
- **Recency** breaks at 90 days: last purchase within 90 days -> about 70% repeat; longer ago -> 30-40%.
- **Purchases in the last 90 days:** steady rise from 38% (none) to 100% (10+).
- **Value of the current purchase:** small effect (between 52% and 67%).

In [292]:
# Does the customer's usual rhythm predict the outcome? Compare the typical gap with the 90-day window
fig = make_subplots(rows=1, cols=2, subplot_titles=("Average gap between past purchases", "Purchases in the previous 365 days"))
for value, color, label in [(0, GREY, "no repeat"), (1, BLUE, "repeat")]:
    subset = df[df[TARGET] == value]
    add_density(fig, np.log1p(subset["prior_mean_gap_days"]), color, label, row=1, col=1)
    add_density(fig, np.log1p(subset["purchase_count_365d"]), color, label, row=1, col=2, showlegend=False)
fig.add_vline(x=np.log1p(90), line_dash="dash", line_color=RED, annotation_text="90 days", row=1, col=1)
fig.update_xaxes(title_text="log(1 + days)", row=1, col=1)
fig.update_xaxes(title_text="log(1 + count)", row=1, col=2)
fig.update_yaxes(title_text="density", row=1, col=1)
fig.update_layout(height=450)
fig.show()

df["gap_below_90"] = df["prior_mean_gap_days"] < 90
print(df.groupby("gap_below_90")[TARGET].agg(["mean", "size"]).round(3))
df = df.drop(columns="gap_below_90")

               mean  size
gap_below_90             
False         0.376  3164
True          0.735  3305


In [293]:
first = df["prior_purchase_count"] == 0
print("Repeat rate, first purchase:     ", round(df.loc[first, TARGET].mean(), 3), f"(n={first.sum()})")
print("Repeat rate, returning customers:", round(df.loc[~first, TARGET].mean(), 3), f"(n={(~first).sum()})")

Repeat rate, first purchase:      0.333 (n=603)
Repeat rate, returning customers: 0.583 (n=5866)


### Key insight - Section 8

**Found**
- **Frequency** is the strongest signal: the more past purchases (in total and in the last 90 days), the more likely a repeat.
- **Recency:** a last purchase within 90 days repeats about 70%; longer ago, 30-40%.
- 1st purchases repeat 33%, returning customers 58%.
- Customers whose average gap is under 90 days repeat 74%, the others 38%.
- The value of the current purchase has only a small effect.

**For preprocessing**
- These patterns are turned into features in notebook 02 (`is_first_purchase`, "is the customer late compared with their usual rhythm?").
- Which columns predict the target best, and which are near copies of each other, is also checked there.

## 9. What drives how much they spend?

In [294]:
returned = df[(df[TARGET] == 1) & df[SPEND].notna()].copy()
returned["log_spend"] = np.log1p(returned[SPEND])

spend_cols = ["spend_365d", "prior_mean_purchase_value", "Total a Pagar"]
rhos = [spearmanr(returned[col], returned[SPEND], nan_policy="omit")[0] for col in spend_cols]
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{col}  (rho = {rho:.2f})" for col, rho in zip(spend_cols, rhos)])
for i, col in enumerate(spend_cols, start=1):
    fig.add_scatter(x=np.log1p(returned[col]), y=returned["log_spend"], mode="markers",
                    marker=dict(size=4, color=BLUE, opacity=0.25), showlegend=False, row=1, col=i)
    fig.update_xaxes(title_text=f"log(1 + {col})", row=1, col=i)
fig.update_yaxes(title_text="log(1 + spend_next_90d)", row=1, col=1)
fig.update_layout(height=480)
fig.show()

In [295]:
labels = ["0", "1", "2", "3", "4-5", "6-10", "10+"]
groups = pd.cut(returned["purchase_count_90d"], [-1, 0, 1, 2, 3, 5, 10, 30], labels=labels)
box_data = returned.assign(group=groups).dropna(subset=["group"])
fig = px.box(box_data, x="group", y=SPEND, points=False, log_y=True,
             category_orders={"group": labels}, color_discrete_sequence=[BLUE])
fig.update_layout(title="Frequent buyers return AND spend more",
                  xaxis_title="purchases in the previous 90 days", yaxis_title="spend_next_90d (EUR, log)")
fig.show()

### Key insight - Section 9

**Found**
- How much a customer spends next depends mostly on **past spend** (`spend_365d`, correlation 0.60).
- The value of the current purchase matters much less.
- Frequent buyers return **and** spend more.
- Spend ranges from about 1 EUR to 25,600 EUR.

**For preprocessing**
- Regression on `log1p(spend)`; also test `log1p` on the spend features for linear models.
- On the raw scale, a few big customers dominate errors such as RMSE. Evaluate with the metric Kaggle uses.
- Which features to use for the spend model is checked in notebook 02.

## 10. Categorical variables

In [296]:
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.15,
                    subplot_titles=[f"{col} ({df[col].nunique()} levels)" for col in cat_cols])
for i, col in enumerate(cat_cols):
    filled = df[col].fillna("MISSING")
    summary = filled.value_counts().to_frame("n")
    summary["rate"] = df.groupby(filled)[TARGET].mean()
    summary = summary.head(10)
    labels = [s.split()[-1] if s != "MISSING" else "NA" for s in summary.index]
    position = dict(row=i // 3 + 1, col=i % 3 + 1)
    fig.add_bar(x=labels, y=summary["rate"], marker_color=[RED if n < 30 else BLUE for n in summary["n"]],
                text=summary["n"], textposition="outside", showlegend=False,
                hovertemplate="%{x}<br>repeat rate: %{y:.2f}<br>rows: %{text}<extra></extra>", **position)
    fig.add_hline(y=df[TARGET].mean(), line_dash="dash", line_color=ORANGE, **position)
    fig.update_yaxes(range=[0, 1.15], **position)
fig.update_yaxes(title_text="repeat rate", col=1)
fig.update_layout(title="Repeat rate per level (top 10 by size; red = fewer than 30 rows; number = rows)", height=750)
fig.show()

In [297]:
transport_counts = df["transport_method"].value_counts()
print("transport_method levels:", len(transport_counts), "| with < 20 rows:", (transport_counts < 20).sum())
print("Levels in test that never appear in train:")
for col in cat_cols:
    unseen = set(test[col].dropna()) - set(df[col].dropna())
    if unseen:
        print(f"  {col}: {sorted(unseen)}")

transport_method levels: 80 | with < 20 rows: 46
Levels in test that never appear in train:
  payment_type: ['Maybe Tomorrow Payment 1', 'Maybe Tomorrow Payment 5', 'Maybe Tomorrow Payment 6']
  commercial_zone: ['Decorative Dashboard Zone 1', 'Decorative Dashboard Zone 5']
  transport_method: ['Maybe Tomorrow Delivery 54', 'Maybe Tomorrow Delivery 65']
  warehouse: ['Final Excel Warehouse 2']


### Key insight - Section 10

**Found**
- `payment_type` 9: 95% repeat (502 rows), against 53% for the main type 7. Maybe a recurring-payment agreement.
- `salesperson`: Master 1 repeats 62%, Master 4 39%.
- `transport_method`: 80 levels, 46 with fewer than 20 rows. Some large levels are very informative (level 11: 98% of 253 rows).
- `document_series`: no effect (56% in all three).
- `commercial_zone` (97% in one level) and `warehouse` (99.7% in one level) are almost constant.
- test has levels never seen in train in 4 columns.

**For preprocessing**
- Drop `document_series`. `commercial_zone` and `warehouse` carry little information: drop them or keep only a flag for the rare levels.
- Group rare levels (fewer than 20 rows) into "other", especially in `transport_method`.
- Encode with one-hot for the small columns. For `transport_method`, test frequency or target encoding (target encoding fitted inside each fold, to avoid leakage).
- The encoder must accept unknown levels (`OneHotEncoder(handle_unknown="ignore")`), or the pipeline will fail on test.
- Missing values become their own "missing" category.

## 11. Customers and `clientes.csv`

In [298]:
rows_per_customer = df["customer_id"].value_counts()
share = rows_per_customer.sort_values(ascending=False).cumsum() / len(df)
customer_rate = df.groupby("customer_id")[TARGET].mean()

fig = make_subplots(rows=1, cols=3, subplot_titles=("Purchases per customer in train", "Concentration of rows",
                                                    "Repeat rate per customer"))
fig.add_histogram(x=np.log10(rows_per_customer), nbinsx=30, marker_color=BLUE, row=1, col=1)
fig.add_scatter(x=np.arange(1, len(share) + 1) / len(share), y=share.values, mode="lines",
                line=dict(color=BLUE, width=2), row=1, col=2)
fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=GREY, dash="dash"), row=1, col=2)
fig.add_histogram(x=customer_rate, nbinsx=20, marker_color=BLUE, row=1, col=3)

ticks = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
fig.update_xaxes(title_text="rows (log scale)", tickvals=np.log10(ticks), ticktext=ticks, row=1, col=1)
fig.update_xaxes(title_text="share of customers (largest first)", row=1, col=2)
fig.update_xaxes(title_text="share of purchases followed by another", row=1, col=3)
fig.update_yaxes(title_text="customers", row=1, col=1)
fig.update_yaxes(title_text="share of rows", row=1, col=2)
fig.update_yaxes(title_text="customers", row=1, col=3)
fig.update_layout(showlegend=False, height=450)
fig.show()

print("Customers:", len(rows_per_customer), "| with 1 row:", (rows_per_customer == 1).sum(),
      "| top 10% of customers hold", f"{share.iloc[int(len(share) * 0.1)]:.0%}", "of rows")
print("Customers who NEVER repeat within 90 days:", (customer_rate == 0).sum())
print("test rows whose customer is in train:", f"{test['customer_id'].isin(df['customer_id']).mean():.1%}")

Customers: 607 | with 1 row: 157 | top 10% of customers hold 50% of rows
Customers who NEVER repeat within 90 days: 256
test rows whose customer is in train: 94.5%


In [299]:
merged = df.merge(clientes, on="customer_id", how="left")
print("train rows with a match in clientes:", merged["country_or_region"].notna().mean())

top_countries = merged["country_or_region"].value_counts().head(4).index
sub = merged[merged["country_or_region"].isin(top_countries)]
country_rate = sub.groupby("country_or_region")[TARGET].agg(["mean", "size"])
island_rate = merged.groupby(merged["island"].isna())[TARGET].agg(["mean", "size"])

fig = make_subplots(rows=1, cols=2, subplot_titles=("Repeat rate by country", "Repeat rate by whether 'island' is filled"))
fig.add_bar(x=[c.split(":")[0].replace("Country ", "") for c in country_rate.index], y=country_rate["mean"],
            marker_color=BLUE, text=[f"n={n}" for n in country_rate["size"]], textposition="outside", row=1, col=1)
fig.add_bar(x=["island known", "island missing"], y=island_rate["mean"], marker_color=[BLUE, ORANGE],
            text=[f"n={n}" for n in island_rate["size"]], textposition="outside", row=1, col=2)
fig.update_yaxes(range=[0, 1], row=1)
fig.update_yaxes(title_text="repeat rate", row=1, col=1)
fig.update_layout(showlegend=False, height=420)
fig.show()

first_year = df.groupby("customer_id")["purchase_date"].min().dt.year
since = clientes.set_index("customer_id")["customer_since_year"].reindex(first_year.index)
print("customer_since_year vs first purchase year in train:",
      {"earlier": (since < first_year).sum(), "same": (since == first_year).sum(), "later": (since > first_year).sum()})

train rows with a match in clientes: 1.0


customer_since_year vs first purchase year in train: {'earlier': np.int64(197), 'same': np.int64(410), 'later': np.int64(0)}


### Key insight - Section 11

**Found**
- A few customers dominate: median 4 rows per customer, maximum 276, and the top 10% of customers hold 50% of the rows. 157 customers appear only once.
- 256 of the 607 customers never repeat within 90 days.
- 94.5% of the test rows belong to customers already in train.
- In `clientes.csv`, two things are related to the target: the **country** (CT5 repeats 39%, CT10 57%) and whether **`island` is filled** (64% vs 52%). The other columns are anonymised addresses with too many levels. `customer_since_year` agrees with the purchase history.

**For preprocessing**
- Join `clientes.csv` on `customer_id` and add `country` (small countries grouped into "other") and `island_missing`.
- Country CT5 overlaps almost completely with 0% VAT (notebook 02): keep one of the two.
- Drop the other address columns and `customer_type`.

## 12. `random_noise`

In [300]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("random_noise by class", "Repeat rate by random_noise bin"))
for value, color, label in [(0, GREY, "no repeat"), (1, BLUE, "repeat")]:
    add_density(fig, df.loc[df[TARGET] == value, "random_noise"].dropna(), color, label, row=1, col=1)
rate_by_bins(df, "random_noise", [-5, -1.5, -0.5, 0, 0.5, 1.5, 5], fig=fig, row=1, col=2)
fig.update_layout(title=f"random_noise has no relation with the target<br><sup>dashed line = overall rate ({df[TARGET].mean():.2f})</sup>",
                  height=500, margin_t=120)
fig.show()

from scipy.stats import mannwhitneyu, shapiro
rho, p_rho = spearmanr(df["random_noise"], df[TARGET], nan_policy="omit")
u, p_u = mannwhitneyu(df.loc[df[TARGET] == 1, "random_noise"].dropna(), df.loc[df[TARGET] == 0, "random_noise"].dropna())
print(f"Spearman with target: {rho:.3f} (p = {p_rho:.2f}) | Mann-Whitney p = {p_u:.2f}")
print(f"Spearman with spend (returners): {spearmanr(returned['random_noise'], returned[SPEND], nan_policy='omit')[0]:.3f}")
print("Mean / std:", round(df["random_noise"].mean(), 3), "/", round(df["random_noise"].std(), 3))

Spearman with target: -0.008 (p = 0.51) | Mann-Whitney p = 0.51
Spearman with spend (returners): 0.011
Mean / std: -0.022 / 1.001


### Key insight - Section 12

**Found:** `random_noise` is random (mean 0, std 1) and unrelated to both targets (correlation about 0, p = 0.51).

**For preprocessing:** drop it (done in `clean_transactions`).

## 13. Is test similar to train?

In [301]:
compare_cols = ["customer_tenure_days", "prior_purchase_count", "days_since_previous_purchase", "Total a Pagar"]
fig = make_subplots(rows=1, cols=4, subplot_titles=compare_cols)
for i, col in enumerate(compare_cols, start=1):
    for data, color, label in [(df, BLUE, "train"), (test, ORANGE, "test")]:
        values = np.sort(np.log1p(data[col].dropna().clip(lower=0)))
        fig.add_scatter(x=values, y=np.arange(1, len(values) + 1) / len(values), mode="lines",
                        line=dict(color=color, width=2), name=label, legendgroup=label, showlegend=(i == 1),
                        row=1, col=i)
    fig.update_xaxes(title_text="log(1 + value)", row=1, col=i)
fig.update_yaxes(title_text="cumulative share", row=1, col=1)
fig.update_layout(title="Distribution of key features: train vs test", height=430)
fig.show()

print("First purchases - train:", f"{(df['prior_purchase_count'] == 0).mean():.1%}", "| test:", f"{(test['prior_purchase_count'] == 0).mean():.1%}")
pd.DataFrame({"train median": df[compare_cols].median(), "test median": test[compare_cols].median()}).round(1)

First purchases - train: 9.3% | test: 2.2%


,train median,test median
customer_tenure_days,918.0,2662.0
prior_purchase_count,11.0,28.0
days_since_previous_purchase,57.0,59.0
Total a Pagar,476.7,663.9


### Key insight - Section 13

**Found**
- Test customers are older and more loyal: median tenure 2,662 vs 918 days, 28 vs 11 prior purchases, and only 2.2% first purchases vs 9.3%. Recency is similar.
- So test will probably have a higher repeat rate than train.

**For preprocessing and validation**
- Validate on the most recent part of train: it is the most similar to test.
- Check the decision threshold on that recent validation period, not on the whole of train.

## 14. Cleaning function and summary

All the fixes from Sections 2, 3 and 12 in one function, to move to `src/preprocessing.py`. Removing rows and rebuilding the -4 gaps from dates happen in train only.

In [302]:
def clean_transactions(data, is_train=True):
    """Rule-based cleaning from Sections 2, 3 and 12. Learns nothing from the data."""
    out = data.copy()
    out["purchase_date"] = pd.to_datetime(out["purchase_date"])
    predictors = [c for c in out.columns if c not in ["ID", "customer_id", TARGET, SPEND]]

    if is_train:
        out = out[~out[predictors].isna().all(axis=1)]                     # 2. empty rows
        out = out[~out.drop(columns="ID").duplicated()]                     # 2. duplicates

    has_history = out["prior_purchase_count"] > 0                           # 3.1 mean = total / count
    out["prior_mean_purchase_value"] = (out["prior_total_spend"] / out["prior_purchase_count"]).where(has_history)

    max_too_big = out["prior_max_purchase_value"] > out["prior_total_spend"] + 0.01      # 3.2 max > total
    if is_train:                                                            # 3.2 rebuild from earlier purchases in train
        out.loc[max_too_big, "prior_max_purchase_value"] = rebuild_max(out)[max_too_big]
    else:
        out.loc[max_too_big, "prior_max_purchase_value"] = np.nan

    out["spend_365d"] = out["spend_365d"].abs()                             # 3.3 sign flips
    valid = (out["spend_365d"] >= out["spend_180d"] - 0.01) & (out["spend_365d"] <= out["prior_total_spend"] + 0.01)
    if is_train:                                                            # 3.3 rebuild from dates in train
        wrong = out["spend_365d"].notna() & ~valid                          #     (NaN when a purchase is hidden)
        out.loc[wrong, "spend_365d"] = rebuild_spend_365d(out)[wrong]
    else:
        out.loc[~valid, "spend_365d"] = np.nan                              # 3.3 sentinel / inflated

    if is_train:                                                            # 3.4 -4 in days_since: real gap from dates
        ordered = out.sort_values(["customer_id", "purchase_date"])
        gap = ordered.groupby("customer_id")["purchase_date"].diff().dt.days
        fill = gap.index[(ordered["days_since_previous_purchase"] < 0) & gap.notna()]
        out.loc[fill, "days_since_previous_purchase"] = gap.loc[fill]
        minus4 = out["prior_mean_gap_days"] < 0                             # 3.4 -4 in mean gap: rebuild from dates
        out.loc[minus4, "prior_mean_gap_days"] = rebuild_mean_gap(out)[minus4]
    for col in ["days_since_previous_purchase", "prior_mean_gap_days"]:     # 3.4 remaining -4 -> NaN
        out.loc[out[col] < 0, col] = np.nan

    return out.drop(columns="random_noise")                                 # 12.


train_clean = clean_transactions(train, is_train=True)
test_clean = clean_transactions(test, is_train=False)
print("train:", train.shape, "->", train_clean.shape)
print("test: ", test.shape, "->", test_clean.shape)

# Same result as the step-by-step cleaning above (apart from the recovered spend target and random_noise)
same_cols = [c for c in train_clean.columns if c != SPEND]
print("Matches the step-by-step version:", train_clean[same_cols].reset_index(drop=True).equals(df[same_cols].reset_index(drop=True)))

train: (6534, 40) -> (6469, 39)
test:  (986, 38) -> (986, 37)
Matches the step-by-step version: True


In [303]:
# Save the cleaned data for notebook 02 (run this notebook first)
# train_clean also gets the 176 spend values recovered in Section 6 (target only, never a feature)
from pathlib import Path

train_clean[SPEND] = df.loc[train_clean.index, SPEND]
Path(f"{DATA_DIR}/processed").mkdir(exist_ok=True)
train_clean.to_csv(f"{DATA_DIR}/processed/train_clean.csv", index=False)
test_clean.to_csv(f"{DATA_DIR}/processed/test_clean.csv", index=False)
print("Saved:", train_clean.shape, test_clean.shape,
      "| returners without spend:", ((train_clean[TARGET] == 1) & train_clean[SPEND].isna()).sum())

Saved: (6469, 39) (986, 37) | returners without spend: 20


### Summary of decisions

| # | Issue | Rows | Decision | Where |
|---|---|---|---|---|
| 2 | Empty rows (only ID, customer, target) | 65 | drop | train only |
| 3.1 | `prior_mean_purchase_value`: `1e9`, sign flip, too large, missing | 272 | recompute total / count | train + test |
| 3.2 | `prior_max_purchase_value` > total spend | 45 | rebuild from earlier purchases (train); NaN, impute (test) | train + test |
| 3.3 | `spend_365d`: `1e9`, sign flip, > total spend | 69 | remove sign; rebuild from dates (train), else NaN, impute | train + test |
| 3.4 | -4 in `days_since_previous_purchase` | 16 | rebuild from dates (train); NaN, impute (test) | train + test |
| 3.4 | -4 in `prior_mean_gap_days` | 16 | rebuild from dates (train); NaN, impute (test) | train + test |
| 5 | NaN because there is no history (1st / 2nd / 3rd purchase) | 603 / 445 / 365 | flag + constant | pipeline |
| 6 | Returners without spend | 196 | recover 176 from train, drop 20 | regression target |
| 10 | Rare levels; levels seen only in test | 46 rare + 8 unseen | group rare, accept unknown | pipeline |
| 12 | `random_noise` | all | drop | train + test |

**Next steps (pipeline)**
- Validation: **time-based**, not random K-fold.
- Imputation, encoding and scaling inside the sklearn `Pipeline`, fitted on train only.
- Regression on `log1p(spend)`.
- New features and feature selection: notebook `02_feature_engineering_selection.ipynb`, which loads the cleaned data saved below.